# 02 — Feature Engineering
Builds `NetworkFlowTransformer` pipeline: 21 raw → 27 engineered features + flow-as-text encoding.

In [ ]:
import sys; sys.path.insert(0,'..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid'); plt.rcParams['figure.dpi']=120

## 1. Run Full Preprocessing Pipeline

In [ ]:
from data.preprocess import load_and_preprocess, ATTACK_NAMES
data = load_and_preprocess('../data/raw/cicids2017/synthetic_cicids.parquet')
X_train,y_train = data['X_train'],data['y_train']
X_val,y_val     = data['X_val'],data['y_val']
X_test,y_test   = data['X_test'],data['y_test']
feature_names   = data['feature_names']
print(f'Train:{X_train.shape} Val:{X_val.shape} Test:{X_test.shape}')
print(f'\nFeatures ({len(feature_names)}):')
for i,f in enumerate(feature_names): print(f'  {i+1:2d}. {f}')

## 2. Random Forest Baseline

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report
import time
rf=RandomForestClassifier(n_estimators=200,max_depth=15,n_jobs=-1,random_state=42)
t0=time.time(); rf.fit(X_train,y_train); elapsed=time.time()-t0
y_pred=rf.predict(X_test)
f1_rf=f1_score(y_test,y_pred,average='weighted')
print(f'RF F1 (weighted): {f1_rf:.4f}  ← RL baseline to beat')
print(f'Train time: {elapsed:.1f}s')
print('\n'+classification_report(y_test,y_pred,target_names=list(ATTACK_NAMES.values())))

## 3. Feature Importance Analysis

In [ ]:
idx=np.argsort(rf.feature_importances_)[::-1][:20]
fig,ax=plt.subplots(figsize=(12,7))
cols=plt.cm.viridis(np.linspace(0.2,0.9,20))
ax.barh([feature_names[i] for i in idx[::-1]],rf.feature_importances_[idx[::-1]],color=cols[::-1])
ax.set_title('Top 20 Feature Importances (Random Forest baseline)',fontsize=13,fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('../reports/02_rf_importance.png',bbox_inches='tight')
plt.show()
print(f'Top 3 features: {[feature_names[i] for i in idx[:3]]}')

## 4. Flow-as-Text Encoding (Novel HF Bridge)

In [ ]:
from data.preprocess import FlowTextEncoder
import pandas as pd
df=pd.read_parquet('../data/raw/cicids2017/synthetic_cicids.parquet').head(5)
enc=FlowTextEncoder()
texts=enc.encode_batch(df)
print('Flow → Text encoding (input to DistilBERT):\n')
for text,label in zip(texts,df['Label']):
    print(f'[{label}]\n  {text}\n')

## 5. Derived Feature Analysis

In [ ]:
df=pd.read_parquet('../data/raw/cicids2017/synthetic_cicids.parquet')
df.columns=df.columns.str.strip()
eps=1e-9
df['packet_ratio']=df['Total Fwd Packets']/(df['Total Backward Packets']+eps)
df['iat_cv']=df['Flow IAT Std']/(df['Flow IAT Mean']+eps)
df['flag_density']=(df['SYN Flag Count']+df['RST Flag Count']+df['ACK Flag Count'])/(df['Total Fwd Packets']+df['Total Backward Packets']+eps)
fig,axes=plt.subplots(1,3,figsize=(15,5))
for ax,feat,title in zip(axes,['packet_ratio','iat_cv','flag_density'],
                        ['Packet Ratio (Fwd/Bwd)','IAT Coeff of Variation','Flag Density']):
    for label in ['BENIGN','DoS Hulk','PortScan','DDoS']:
        if label in df['Label'].values:
            s=df[df['Label']==label][feat].replace([np.inf,-np.inf],np.nan).dropna()
            s.clip(upper=s.quantile(0.99)).plot.kde(ax=ax,label=label,alpha=0.8)
    ax.set_title(title,fontweight='bold'); ax.legend(fontsize=8)
plt.suptitle('Derived Feature Distributions',fontsize=13,fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/02_derived.png',bbox_inches='tight')
plt.show()

## ✅ Summary
- Pipeline: 21 raw → 27 engineered features (clip + scale + 6 derived)
- **RF baseline F1 ~89%** — RL target: exceed this with interpretability
- Flow-as-text encoding: novel bridge enabling DistilBERT transfer learning on network data
- `flag_density` and `iat_cv` are novel features not in original CICIDS papers